### **Jupyter Notebook By Shiqi Liu For CICC Quant Intern**

#### **qpc syntax templete**

* Init("/qp/data/ini/cn.eq.base.ini")

* set_now(20230718, 1500)

* data_dictionary()

* mkbin -n factor_name -s 20160101 -e 20230721 -u estu.a

* factorperf -n factor_name -s 20150101 -e 20230721 -u estu.a -U estu.a -d y

* DateTimeSpec(Constants.Yesterday, TimeSpec(1500))

* /mnt/winsrv/xuzhitao/tmp/liushiqi

* /qp/3rd/anaconda3/bin/pip install package_name==3.0.0 -i http://10.51.129.74/artifactory/api/pypi/public-pypi-virtual/simple/ --trusted-host 10.51.129.74 -user

#### **File System Structure 文件系统结构**
```markdown-tree
- /
    - home/
        - xuzhitao/
        - sunsy/
        - liuzixuan/
        - zhangyiming/
        - liushiqi/
            - git/
                - fh/
                    - bins/
                        - formal/
                            - estu.a/
                                - amihud.bin
                                - real_skewlarge.bin
                                - real_skewlarge_weekly.bin
                                ...
                            - estu.b/
                                ...
                        - ini/
                            - factor.dict.estu.a.ini
                            - factor.dict.estu.b.ini
                            - user.dict.ini
                        - tmp/
                            - estu.a/
                                - amihud.bin
                                - real_skewlarge.bin
                                - real_skewlarge_weekly.bin
                                ...
                            - estu.b/
                                ...
                    - dbins/
                        - ini/
                            - factor.dict.estu.a.ini
                            - factor.dict.estu.b.ini
                    - research/
                        - algo/
                            - amihud/
                                - algo.py
                                - postjob.sh
                                - prejob.sh
                                - README.md
                                - configs/
                                    - algo.ini
                                    - eval.ini
                                - output/
                                    - figures/
                                        - estu.a/
                                            - cover.ratio.png
                                            - group.ret.png
                                            - ma.png
                                            - rankic.png
                                    - log/
                                        - amihud.log
                                    - perf/
                                        - estu.a/
                                    - reports/
                                        - estu.a/
                                            - amihud.perf.is.docx
                                    - summaries/
                                - __pycache__/
                                    - algo.cpython-38.pyc
                            - min_dev4/
                                ...
                            - stability/
                                ...
                            - stability_monthly/
                                ...
                            - amihud_stability_monthly/
                                ...
                            ...
                        - backup/
                            - bp/
                                - 20230719.104104/
                                    - algo.py
                                    - postjob.sh
                                    - prejob.sh
                                    - README.md
                                    - configs/
                                        - algo.ini
                                        - evel.ini
                                        - configs/ 
                                            - algo.ini
                                            - eval.ini
                                        
                                    - output/
                                        - figures/
                                        - summaries/
                                ...
                        - common/
                        - output/
                    - dresearch/
                        - algo/
                        - backup/
                        - common/
                        - output/
                    - submit/
                        - algo/
                        - common/
                    - dsubmit/
                        - algo/
                        - common/
                    - zoo/
                    - path.ini
        ...
    - mnt/
        - winsrv/
            - xuzhitao/
                - tmp/
                    - liushiqi/
                        - test.py
                        - backup.py
            ...
                ...
                    ...

#### **Factor Algorithms 因子算法复现与改进**

In [1]:
# Import Libraries and Modules
from qpc import *
import pdb
import numpy as np
import pandas as pd
from scipy.linalg import svd

# 所有非平滑回测的年份为20150101-20230721；所有进行平滑的回测的年份为20160101-20230721
# 所有回测的universe为estu.a (全A股)

### 研报一 ###

# roe_growth: IC = 1.57
class roe_growth(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.qfa_roe = QPData("fundamental_finind_s_qfa_roe")

    def trade(self):
        self.print_soy()
        # Algo logic:
        # 1/ data preprocessing (finantialize)
        # 2/ calculate variable
        # 3/ apply zscore
        # 4/ neutralize
        # 5/ store factor
        roeMat = self.qfa_roe.daily_matrix_pd(Constants.Yesterday)
        roeMat = roeMat.ffill(axis = 1)
        roeMat = roeMat.bfill(axis = 1)
        ROE_q = roeMat
        ROE_yoy = ROE_q[:, 0] - ROE_q[:, 4]
        ROE_stb = ROE_q.iloc[:, [0, 4, 8, 12]].mean(axis = 1) / ROE_q.iloc[:, [0, 4, 8, 12]].std(axis = 1)
        ROE_last = 1 / 12 * ROE_q.iloc[:, 0:12].sum(axis = 1)
        ROE_growth = ROE_q[:, 0] + ROE_q[:, 8] - ROE_q[:, 4] * 2
        def formater(para):
            para = para.values
            para = self.neutralize(para ,style=['indzx','barra'])
            para[~np.isfinite(para)] = np.nan
            para = zscore(para, winsorize_method='mad')
            if np.isnan(para).astype(int).sum() == para.shape[0]:
                para = zscore(para, winsorize_method = '3sigma')
            return para
        ROE_yoy = formater(ROE_yoy)
        ROE_stb = formater(ROE_stb)
        ROE_last = formater(ROE_last)
        ROE_growth = formater(ROE_growth)
        ROE_enhance = 1 / 4 * (ROE_yoy + ROE_stb + ROE_last + ROE_growth)
        # store factor
        store_daily_tradable_vector(self.field_for_store, ROE_enhance)


# roe_enhance: IC = 1.40
class roe_enhance(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.qfa_roe = QPData("fundamental_finind_s_qfa_roe")
        # Your Configs

    def trade(self):
        self.print_soy()
        # Algo logic:
        # 1/ data preprocessing (finantialize)
        # 2/ calculate variable
        # 3/ apply zscore
        # 4/ neutralize
        # 5/ store factor
        roeMat = self.qfa_roe.daily_matrix_pd(Constants.Yesterday)
        roeMat = roeMat.ffill(axis = 1)
        roeMat = roeMat.bfill(axis = 1)
        ROE_q = roeMat
        ROE_yoy = ROE_q.iloc[:, 0] - ROE_q.iloc[:, 4]
        ROE_stb = ROE_q.iloc[:, [0, 4, 8, 12]].mean(axis = 1) / ROE_q.iloc[:, [0, 4, 8, 12]].std(axis = 1)
        ROE_last = 1 / 12 * ROE_q.iloc[:, 0:12].sum(axis = 1)
        ROE_growth = ROE_q.iloc[:, 0] + ROE_q.iloc[:, 8] - ROE_q.iloc[:, 4] * 2
        def formater(para, check):
            para = para.values
            if check == 0:
                num = 1
                # para = self.neutralize(para ,style=['indzx','barra'])
            para[~np.isfinite(para)] = np.nan
            para = zscore(para, winsorize_method='mad')
            if np.isnan(para).astype(int).sum() == para.shape[0]:
                para = zscore(para, winsorize_method = '3sigma')
            return pd.DataFrame(para)
        ROE_yoy = formater(ROE_yoy, 0)
        ROE_stb = formater(ROE_stb, 1)
        ROE_last = formater(ROE_last, 0)
        ROE_growth = formater(ROE_growth, 0)
        ROE_test = pd.concat([ROE_yoy, ROE_stb, ROE_last, ROE_growth], axis = 1)
        ROE_enhance = ROE_test.mean(axis = 1).values
        # store factor
        store_daily_tradable_vector(self.field_for_store, ROE_enhance)



### 研报二 ###

# min_dev3 高频偏度:
class min_dev3(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.min_price = QPData('close.b1')
        self.open = QPData('open')

    def trade(self):
        self.print_soy()
        N = 240
        close_price = self.min_price.history_matrix_pd(Constants.Yesterday).iloc[:, 0]
        open_price = self.open.history_matrix_pd(1)
        new_df = self.min_price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        new_df.insert(0, 'close_p', open_price)
        df_diff_nomi = new_df.pct_change(axis = 1).iloc[:, 1:].pow(3).sum(axis = 1) * ((N) ** 0.5)
        df_diff_deno = new_df.pct_change(axis = 1).iloc[:, 1:].pow(2).sum(axis = 1).pow(1.5)
        store_daily_tradable_vector(self.field_for_store, (df_diff_nomi / df_diff_deno).to_numpy())

# min_dev4 高频偏度 月平滑: IC = 1.68
class min_dev4(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        # Your Configs

    def trade(self):
        self.print_soy()
        bv = self.min_dev3.history_matrix(20)
        bp = np.mean(bv, axis = 1, keepdims = True)
        # z = zscore(bp,winsorize_method='mad')
        # z = self.neutralize(z,style=['indzx','barra'])
        store_daily_tradable_vector(self.field_for_store, bp)

# xxbd 下行波动
class xxbd(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.min_price = QPData('close.b1')
        self.open = QPData('open')

    def trade(self):
        self.print_soy()
        N = 240
        open_price = self.open.history_matrix_pd(1)
        new_df = self.min_price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        new_df.insert(0, 'open_p', open_price)
        df_diff_nomi = new_df.pct_change(axis = 1).iloc[:, 1:].applymap(lambda x:0 if x>0 else x).pow(2).sum(axis = 1) * ((N) ** 0.5)
        df_diff_deno = new_df.pct_change(axis = 1).iloc[:, 1:].pow(2).sum(axis = 1)
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, (df_diff_nomi / df_diff_deno).to_numpy())
        
# xxbd_monthly 下行波动 月平滑: IC = 0.63
class xxbd_monthly(TradeRule):
    def __init__(self,value):
         TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.xxbd.history_matrix(20)
        bp = np.mean(bv, axis = 1, keepdims = True)
        # z = zscore(bp,winsorize_method='mad')
        # z = self.neutralize(z,style=['indzx','barra'])
        store_daily_tradable_vector(self.field_for_store, -bp)


# wpcj 尾盘成交
class wpcj(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.volume_30 = QPData('volume.b30')

    def trade(self):
        self.print_soy()
        N = 240
        vol_daily_end = self.volume_30.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).iloc[:, -1]
        vol_daily_total = self.volume_30.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).sum(axis = 1)
        store_daily_tradable_vector(self.field_for_store, (vol_daily_end / vol_daily_total).to_numpy())
        
# wpcj_monthly 尾盘成交 月平滑: IC = 0.24
class wpcj_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.wpcj.history_matrix(20)
        bp = np.mean(bv, axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, bp)
 
# gplj 高频量价
class gplj(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.volume = QPData("volume.b1")
        self.price = QPData('close.b1')

    def trade(self):
        self.print_soy()
        price = self.price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        volume = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).replace([np.nan, np.inf, -np.inf], 0)
        total_volume = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).sum(axis = 1)
        corr = []
        volume_ratio = volume.divide(total_volume.values, axis = 0).replace([np.inf, -np.inf, np.nan], 0)
        for i in range(volume_ratio.shape[0]):
            corr.append(np.corrcoef(volume_ratio.iloc[i, :], price.iloc[i, :])[0, 1])
        # volume_ratio = volume.divide(total_volume.values, axis = 0).replace([np.inf, -np.inf, np.nan], 0)
        # corr = price.apply(lambda row: pearsonr(row, volume_ratio.loc[row.name])[0], axis = 1)
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, np.array(corr))

# gplj_monthly 高频量价 月平滑: IC = 3.21
class gplj_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.gplj.history_matrix(20)
        bp = np.mean(bv, axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, -bp)



# gjfz 改进反转
class gjfz(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.min_price = QPData('close.b1')

    def trade(self):
        self.print_soy()
        N = 240
        nomi = self.min_price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).iloc[:, -1]
        deno = self.min_price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).iloc[:, 29]
        store_daily_tradable_vector(self.field_for_store, (nomi / deno).to_numpy())

# gjfz_monthly 改进反转 月平滑: IC = 4.00
class gjfz_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.gjfz.history_matrix(20)
        bp = bv.prod(axis = 1) - 1
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, -bp)


# min_enhance 研报二中四个因子的等权组合因子: IC = 3.81
class min_enhance(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.price = QPData('close.b1')

    def trade(self):
        self.print_soy()
        p_open = self.price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).iloc[:, 0]
        p_close = self.price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).iloc[:, -1]
        factor_1 = self.min_dev4.history_matrix_pd(2)
        factor_2 = self.xxbd_monthly.history_matrix_pd(2)
        factor_3 = self.wpcj_monthly.history_matrix_pd(2)
        factor_4 = self.gplj_monthly.history_matrix_pd(2)
        factor_5 = self.gjfz_monthly.history_matrix_pd(2)
        df_train = pd.concat([factor_1.iloc[:, 0], factor_2.iloc[:, 0], factor_3.iloc[:, 0], factor_4.iloc[:, 0], factor_5.iloc[:, 0]], axis = 1)
        df_train = df_train.replace([np.inf, -np.inf], 0)
        df_train = df_train.fillna(0)
        # model = Ridge(alpha = 1)
        # model.fit(df_train , (p_close - p_open).fillna(0))
        # coef = model.coef_
        df_test = pd.concat([factor_1.iloc[:, 1], factor_2.iloc[:, 1], factor_3.iloc[:, 1], factor_4.iloc[:, 1], factor_5.iloc[:, 1]], axis = 1)
        # z = df_test.apply(lambda row: np.dot(row, coef), axis = 1).fillna(0).to_numpy()
        # pdb.set_trace()
        z2 = (0.2 * df_test).sum(axis = 1).to_numpy()
        z3 = 0.2 * (df_test.apply(lambda x: pow(x, 3)).sum(axis = 1).to_numpy())
        store_daily_tradable_vector(self.field_for_store, z3)



### 研报三 ###

# real_skew 已实现偏度
class real_skew(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.min_price = QPData('close.b1')
        self.open = QPData('open')

    def trade(self):
        self.print_soy()
        new_df = self.min_price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        r_mean = new_df.pct_change(axis = 1).iloc[:, 1:].sum(axis = 1) / 239
        df_diff_nomi = new_df.pct_change(axis = 1).iloc[:, 1:].sub(r_mean, axis = 0).pow(3)
        real_var = (1 / 238) * new_df.pct_change(axis = 1).iloc[:, 1:].sub(r_mean, axis = 0).pow(2).sum(axis = 1)
        df_diff_deno = real_var.pow(1.5)
        z = (1 / 239) * (df_diff_nomi.sum(axis = 1) / df_diff_deno).to_numpy()
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, z)
        
# real_skew_weekly 已实现偏度 周平滑: IC = 4.43
class real_skew_weekly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.real_skew.history_matrix(5)
        bp = np.mean(bv, axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, -bp)

# real_skewlarge 大量成交已实现偏度（*此算法因理解错误 代码逻辑不准确！）
class real_skewlarge(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.min_price = QPData('close.b1')
        self.open = QPData('open')
        self.volume = QPData('volume.b1')

    def trade(self):
        self.print_soy()
        new_df = self.min_price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        r_mean = new_df.pct_change(axis = 1).iloc[:, 1:].sum(axis = 1) / 239
        df_diff_nomi = new_df.pct_change(axis = 1).iloc[:, 1:].sub(r_mean, axis = 0).pow(3)
        real_var = (1 / 238) * new_df.pct_change(axis = 1).iloc[:, 1:].sub(r_mean, axis = 0).pow(2).sum(axis = 1)
        df_diff_deno = real_var.pow(1.5)
        z = ((1 / 239) * (df_diff_nomi.sum(axis = 1) / df_diff_deno).to_numpy()).reshape(-1, 1)
        volume_min = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        volume_total = pd.DataFrame(volume_min.sum(axis = 1))
        volume_total.rename(columns={volume_total.columns[0]: 'total_v'}, inplace = True)
        threshold = volume_total['total_v'].quantile(2/3)
        volume_total['rank'] = volume_total['total_v'].apply(lambda x: 1 if x > threshold else np.nan)
        z = (volume_total['rank'].to_numpy().reshape(-1, 1) * z)
        z = pd.DataFrame(z).replace([np.inf, -np.inf], np.nan).to_numpy()
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, z)

# real_skewlarge_weekly 大量成交已实现偏度 周平滑: IC = 5.02
class real_skewlarge_weekly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.real_skewlarge.history_matrix(5)
        bp = np.mean(bv, axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, -bp)


# reth8 尾盘半小时收益率:
class reth8(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.price = QPData('close.b30')

    def trade(self):
        self.print_soy()
        df = self.price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        z = (df.iloc[:, -1] / df.iloc[:, -2]).to_numpy() - 1
        store_daily_tradable_vector(self.field_for_store, z)

# reth8_weekly 尾盘半小时收益率 周平滑: IC = 2.23
class reth8_weekly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.reth8.history_matrix(5)
        bp = np.mean(bv, axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, -bp)

# ret_intraday 日内收益率
class ret_intraday(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.open = QPData('open')
        self.close = QPData('close.b1')

    def trade(self):
        self.print_soy()
        open_p = self.open.history_matrix_pd(1)
        close_p = self.close.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).iloc[:, -1]
        close_p = pd.DataFrame(close_p)
        z = (close_p.to_numpy() / open_p.to_numpy()) - 1
        z[~np.isfinite(z)] = np.nan
        pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, z)

# ret_intraday_monthly 日内收益率 月平滑: IC = 0.70
class ret_intraday_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        # Your Configs

    def trade(self):
        self.print_soy()
        bv = self.ret_intraday.history_matrix(20)
        bp = np.mean(bv, axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, -bp)

# corr_vrlag 量与滞后收益率相关性
class corr_vrlag(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.volume = QPData("volume.b1")
        self.price = QPData('close.b1')
        self.open = QPData('open')

    def trade(self):
        self.print_soy()
        price = self.price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        volume = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).replace([np.nan, np.inf, -np.inf], 0)
        total_volume = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).sum(axis = 1)
        open_price = self.open.history_matrix_pd(1)
        new_df = price
        new_df.insert(0, 'open_price', open_price)
        r = price.pct_change(axis = 1).iloc[:, 1:-1]
        corr = []
        volume = volume.replace([np.inf, -np.inf, np.nan], 0)
        r = r.replace([np.inf, -np.inf, np.nan], 0)
        for i in range(volume.shape[0]):
            corr.append(np.corrcoef(volume.iloc[i, 1:], r.iloc[i, :])[0, 1])
        # volume_ratio = volume.divide(total_volume.values, axis = 0).replace([np.inf, -np.inf, np.nan], 0)
        # corr = price.apply(lambda row: pearsonr(row, volume_ratio.loc[row.name])[0], axis = 1)
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, np.array(corr))
        
# corr_vrlag_monthly 量与滞后收益率相关性 月平滑: IC = 4.33
class corr_vrlag_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.corr_vrlag.history_matrix(20)
        # pdb.set_trace()
        bp = np.mean(bv , axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, -bp)
        

# corr_vplarge 大成交量价量相关性（*此算法因理解错误 代码逻辑不准确！）: 
class corr_vplarge(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.volume = QPData("volume.b1")
        self.price = QPData('close.b1')

    def trade(self):
        self.print_soy()
        price = self.price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        volume = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        total_volume = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500))).sum(axis = 1)
        corr = []
        total_volume = pd.DataFrame(total_volume)
        # pdb.set_trace()
        for i in range(volume.shape[0]):
            corr.append(np.corrcoef(volume.iloc[i, :], price.iloc[i, :])[0, 1])
        total_volume.rename(columns = {total_volume.columns[0]: 'total_v'}, inplace = True)
        threshold = total_volume['total_v'].quantile(2/3)
        total_volume['rank'] = total_volume['total_v'].apply(lambda x: 1 if x >threshold else np.nan)
        z = (np.array(corr) * total_volume['rank']).to_numpy().reshape(-1, 1)
        z[~np.isfinite(z)] = np.nan
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, z)

# corr_vplarge_weekly 大成交量价量相关性 周平滑: IC = 2.76
class corr_vplarge_weekly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        # Your Configs

    def trade(self):
        self.print_soy()
        bv = self.corr_vplarge.history_matrix(5)
        bp = np.mean(bv, axis = 1, keepdims = True)
        store_daily_tradable_vector(self.field_for_store, -bp)

# ret_open2ah1: (无数据，代码为初始状态)
class ret_open2ah1(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.fundamental_balancesheet_tot_shrhldr_eqy_excl_min_int(Constants.Yesterday)
        cap = fill0(self.cap(Constants.Yesterday),np.nan)
        bp = np_fillna(bv/cap,np.nan)
        # zscore and neutralize
        z = zscore(bp,winsorize_method='mad')
        z = self.neutralize(z,style=['indzx','barra'])
        # zscore and store factor
        store_daily_tradable_vector(self.field_for_store,zscore(z))

# ret_open2al1: (无数据，代码为初始状态)
class ret_open2al1(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.fundamental_balancesheet_tot_shrhldr_eqy_excl_min_int(Constants.Yesterday)
        cap = fill0(self.cap(Constants.Yesterday),np.nan)
        bp = np_fillna(bv/cap,np.nan)
        # zscore and neutralize
        z = zscore(bp,winsorize_method='mad')
        z = self.neutralize(z,style=['indzx','barra'])
        store_daily_tradable_vector(self.field_for_store,zscore(z))
        
# stability 稳定性因子
class stability(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.volume = QPData('volume.b1')

    def trade(self):
        self.print_soy()
        bv = self.volume.daily_matrix(Constants.Yesterday)
        bp = bv.std(axis = 1)
        store_daily_tradable_vector(self.field_for_store, -bp)


# stability_monthly 稳定性因子 月平滑: IC = 3.80
class stability_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        bv = self.stability.history_matrix(20)
        bp = np.nanmean(bv, axis = 1) / np.nanstd(bv, axis = 1)
        bp[~np.isfinite(bp)] = np.nan
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, -bp)


# amihud 非流动性因子: IC = 4.42
class amihud(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.volume = QPData('volume.b1')
        self.price = QPData('close.b1')
        self.open = QPData('open')

    def trade(self):
        self.print_soy()
        open_price = self.open.history_matrix_pd(1)
        p = self.price.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        # p.insert(0, 'close_p', open_price)
        r = p.pct_change(axis = 1).iloc[:, 1:]
        v = self.volume.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        summation = r.abs() / (p.iloc[:, 1:] * v.iloc[:, 1:])
        summation.replace([np.inf, -np.inf], np.nan)
        summation = summation.fillna(0)
        summation = summation.sum(axis = 1)
        z = ((1/239) * summation).to_numpy()
        # Filling NaN, zero, and inf
        # cap = fill0(self.cap(Constants.Yesterday),np.nan)
        # bp = np_fillna(bv/cap,np.nan)
        store_daily_tradable_vector(self.field_for_store, z)

### 多因子自测 ###
# amihud_stability_monthly: IC = 7.02
# 多因子等权组合下，IC可达6.8；手动加权组合下可达7.02；后续采用年化ICIR加权情况下可达7.2
class amihud_stability_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        
        # 导入所有单因子df，并做zscore处理使所有因子处于同一量级
        bv = self.stability.history_matrix(20)
        # bv = zscore(bv)
        stability = -zscore(np.nanmean(bv, axis = 1) / np.nanstd(bv, axis = 1)).reshape(-1, 1)
        amihud = zscore(self.amihud.history_matrix(1))
        real_skew = zscore(self.real_skew_weekly.history_matrix(1))
        corr_vrlag = zscore(self.corr_vrlag_monthly.history_matrix(1))
        gjfz = zscore(self.gjfz_monthly.history_matrix(1))
        gplj = zscore(self.gplj_monthly.history_matrix(1))
        min_dev = zscore(self.min_dev4.history_matrix(1))
        reth8 = zscore(self.reth8_weekly.history_matrix(1))
        ret_intraday = zscore(self.ret_intraday_monthly.history_matrix(1))

        # 从numpy类转为df类以便进行后续处理
        stability_pd = pd.DataFrame(stability)
        amihud_pd = pd.DataFrame(amihud)
        real_skew_pd = pd.DataFrame(real_skew)
        corr_vrlag_pd = pd.DataFrame(corr_vrlag)
        gjfz_pd = pd.DataFrame(gjfz)
        gplj_pd = pd.DataFrame(gplj)
        min_dev_pd = pd.DataFrame(min_dev)
        reth8_pd = pd.DataFrame(reth8)
        ret_intraday_pd = pd.DataFrame(ret_intraday)
        
        # Calculate correlation coefficients
        factor_lst = [stability, amihud, real_skew, corr_vrlag, gjfz, \
        gplj, min_dev, reth8, ret_intraday]
        factor_lst_pd = [stability_pd, amihud_pd, real_skew_pd, corr_vrlag_pd, gjfz_pd, \
        gplj_pd, min_dev_pd, reth8_pd, ret_intraday_pd]
        for k in range(8):
            factor_lst[k] = pd.DataFrame(factor_lst[k]).fillna(0).to_numpy()
            factor_lst_pd[k] = factor_lst_pd[k].replace([np.inf, -np.inf, np.nan], 0)

        for i in range(8):
            for j in range(8 - i):
                print('Correlation between factor', i, 'and factor', i + j, 'is:', \
                (np.corrcoef(pd.DataFrame(factor_lst[i]).iloc[:, 0], pd.DataFrame(factor_lst[i + j]).iloc[:, 0])[0, 1]))

        # Linear combination of factors 线性因子组合: IC = 7.02
        # stability /in 1+-0.25
        # real_skew /in [1, 1.5]
        # corr_vrlag /in [0.5, 1]
        # gjfz /in [1, 1.25]
        # reth8 /in 1+-0.25
        bp = 1.25 * zscore(np.nanmean(bv, axis = 1) / np.nanstd(bv, axis = 1)).reshape(-1, 1) - 1.5 * zscore(amihud) \
        - 1.25 * zscore(real_skew) - 0.75 * zscore(corr_vrlag) \
        - 1.25 * zscore(gjfz) - 1.25 * zscore(reth8) # + zscore(ret_intraday)  - zscore(gplj) - zscore(min_dev)
        bp[~np.isfinite(bp)] = np.nan

        # Orthogonalization:
        # merged_df = pd.concat([stability_pd, amihud_pd, real_skew_pd, corr_vrlag_pd, gjfz_pd, gplj_pd, min_dev_pd, reth8_pd, ret_intraday_pd], axis = 1)
        # merged_df = merged_df.replace([np.inf, -np.inf, np.nan], 0).to_numpy()
        # ortho_df = pd.DataFrame(np.linalg.qr(merged_df)[0])

        store_daily_tradable_vector(self.field_for_store, -bp)



ModuleNotFoundError: No module named 'qpc'

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>研报四 笔记</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 50px;
            color: #333;
        }
        h1, h2, h3 {
            border-bottom: 2px solid #e0e0e0;
            padding-bottom: 10px;
            margin-bottom: 20px;
        }
        ul {
            list-style-type: disc;
            margin-left: 40px;
        }
        li {
            margin-bottom: 10px;
        }
        code {
            background-color: #f4f4f4;
            padding: 2px 4px;
            border-radius: 3px;
        }
        #wrap{
            margin: 30px;
        }
    </style>
</head>
<div id = 'wrap'>
    <body>
        <h1>研报四 笔记</h1>
        <h2>1. 数据处理</h2>
        <h3>1.1 时间重心观察</h3>
        <ul>
            <li>观察每日分频交易涨跌幅绘图。</li>
            <li>提取<code>上涨幅度|上涨时间</code>和<code>下跌幅度|下跌时间</code>。</li>
            <li>计算<code>涨幅时间重心</code>与<code>跌幅时间重心</code>。</li>
            <li>观察涨幅与跌幅时间重心的相关性与density分布。</li>
        </ul>
        <h3>1.2 时间差与时间距离定义</h3>
        <ul>
            <li>定义<code>时间差</code>与<code>时间距离</code>。</li>
            <li>设置平滑基准为月平滑。</li>
            <li>观察目前基础因子的各项指标（如RankIC、Rank ICIR）。</li>
            <li>以跌幅时间重心为因变量，对涨幅时间重心进行回归取residual，月平滑后记为<code>跌幅时间重心偏离因子</code>。</li>
            <li>画图观察此因子与Barra因子等相关性，发现相关性低-->Alpha来源相对独立。</li>
        </ul>
        <h2>2. 解释因子与干扰因子的寻找(时间差Alpha的生效逻辑研究)</h2>
        <h3>2.1 收益率结构的影响</h3>
        <ul>
            <li>隔夜收益率的绝对值越高，涨跌幅时间重心约靠前</li>
            <li>剔除收益率结构的影响（截面归回）。</li>
            <li>回归分两步：日内与隔夜收益回归，涨残差与跌残差再回归，最终残差为Alpha --> 日内收益率及隔夜收益率均为负贡献</li>
            <li>考虑时段1、2与时段7、8的影响（仍旧通过做回归的方式）：
                <ul>
                    <li>时段7、8的涨跌幅为解释因子。</li>
                    <li>时段1、2的涨跌幅为干扰因子。</li>
                </ul>
            </li>
        </ul>
        <h3>2.2 极端样本的影响</h3>
        <ul>
            <li>使用涨跌幅较大的17分钟Bar的平均涨幅和平均跌幅进行回归。</li>
            <li>以全部涨幅和全部跌幅的中位数回归。</li>
                <ul>
                    <li>极端收益率样本的时间中心为解释因子。</li>
                    <li>极端收益率样本的平均涨幅和平均跌幅为干扰因子。</li>
                </ul>
        </ul>
        <h3>2.3 基于地波效应与事件收益纬度的解释</h3>
        <ul>
            <li>解释因子：零涨跌幅数量</li>
            <li>控制变量：盘中是否触及涨跌停</li>
        </ul>
        <h3>2.4 时间差Alpha综合模型</h3>
        <ul>
            <li>解释因子A：时段7的涨跌幅</li>
            <li>解释因子B：时段8的涨跌幅</li>
            <li>解释因子C：日内零涨跌幅分钟的数量</li>
            <li>控制变量：盘中是否触及涨跌停，若是则事件差信号取空，若否则不处理</li>
        </ul>
        <h2>3. 时间差因子选股方案设计</h2>
        <h3>3.1 时间重心偏离（TCD）因子构建</h3>
        <ul>
            <li>观察每日分频交易涨跌幅绘图。</li>
            <li>提取<code>上涨幅度|上涨时间</code>和<code>下跌幅度|下跌时间</code>。</li>
            <li>计算<code>涨幅时间重心</code>与<code>跌幅时间重心</code>。</li>
            <li>观察涨幅与跌幅时间重心的相关性与density分布。</li>
        </ul>
        <h3>3.2 因子相关性分析</h3>
        <h3>3.3 持仓风格分析：整体偏小票样本更有效</h3>
        <h3>3.4 基于多维度信息的因子合成</h3>
    </body>
</div>

In [ ]:
### 研报四 ###
# -*- coding: utf-8 -*-
"""
# Author: liushiqi
# Created Time : Wed Aug  9 10:42:52 CST 2023
# File Name: tcd.py
# Description:
"""
from qpc import *
from sklearn.linear_model import LinearRegression
import pdb
sys.path.append(os.path.join(cur_dir(__file__),'../../common'))

class tcd(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)
        self.price1 = QPData("close.b1")
        self.price30 = QPData('close.b30')
        self.volume1 = QPData('volume.b1')
        self.open = QPData('open')
        self.ret_night = QPData('ret.CC')
        self.ret_day = QPData("ret.Overnight")
    def trade(self):
        self.print_soy()
        price1 = self.price1.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        price30 = self.price30.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        volume1 = self.volume1.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        ret_night = self.ret_night.history_matrix_pd(1)
        ret_day = self.ret_day.history_matrix(1)
        open = self.open.history_matrix_pd(1)

        # Step 1
        new_df_1 = self.price1.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        new_df_2 = self.price30.daily_matrix_pd(DateTimeSpec(Constants.Yesterday, TimeSpec(1500)))
        new_df_1.insert(0, 'open', open)
        new_df_2.insert(0, 'open', open)
        pct_change_df = new_df_1.pct_change(axis = 1).iloc[:, 1:]
        pct_change_df2 = new_df_2.pct_change(axis = 1)

        # define variables (remember to always consider empty list / zero value problem)
        g_u = []
        g_d = []
        r_up_mean = []
        r_down_mean = []
        r_1 = pct_change_df2.iloc[:, 1]
        r_2 = pct_change_df2.iloc[:, 2]
        r_overnight = []

        for j in range(len(price1.index)):
            if price1.index[j] in ret_night.index:
                r_overnight.append(ret_night.iloc[j, 0])
            else:
                r_overnight.append(0)
        r_overnight = pd.DataFrame(r_overnight, columns=['r_overnight'])

        stock_num = price1.shape[0]
        pct_change_df = pct_change_df.replace([np.inf, -np.inf, np.nan], 0)
        for i in range(stock_num):
            up_lst = []
            down_lst = []
            zero_count = 0
            up_sum = 0
            down_sum = 0
            for minutes in range(240):
                if pct_change_df.iloc[i, minutes] > 0:
                    up_lst.append(pct_change_df.iloc[i, minutes])
                    up_sum += minutes * pct_change_df.iloc[i, minutes]
                elif pct_change_df.iloc[i, minutes] < 0:
                    down_lst.append(pct_change_df.iloc[i, minutes])
                    down_sum += minutes * pct_change_df.iloc[i, minutes]
                else:
                    zero_count += 1
            r_up_mean.append(np.array(up_lst).mean() if len(up_lst) > 0 else 0)
            r_down_mean.append(np.array(down_lst).mean() if len(down_lst) > 0 else 0)
            g_u.append(up_sum / np.array(up_lst).sum() if len(up_lst) > 0 else 0)
            g_d.append(down_sum / np.array(down_lst).sum() if len(down_lst) > 0 else 0)
        r_up_mean = pd.DataFrame(r_up_mean, columns = ["r_up_mena"])
        r_down_mean = pd.DataFrame(r_down_mean, columns = ["r_down_mena"])
        g_u = pd.DataFrame(g_u, columns = ['g_u'])
        g_d = pd.DataFrame(g_d, columns = ['g_d'])
        # pdb.set_trace()

        # step 2: regression 将涨跌幅的时间重心单独剥离干扰因子
        # Prepare the data for regression
        r_up_mean = r_up_mean.replace([np.nan, np.inf, -np.inf], 0)
        r_down_mean = r_down_mean.replace([np.nan, np.inf, -np.inf], 0)
        r_1 = r_1.replace([np.nan, np.inf, -np.inf], 0)
        r_2 = r_2.replace([np.nan, np.inf, -np.inf], 0)
        r_overnight = r_overnight.replace([np.nan, np.inf, -np.inf], 0)
        # X_u = pd.concat([r_up_mean, r_1, r_2, r_overnight], axis=1).replace([np.inf, -np.inf, np.nan], 0)
        X_u = np.hstack((r_up_mean.to_numpy().reshape(-1, 1), r_1.to_numpy().reshape(-1, 1), r_2.to_numpy().reshape(-1, $
        y_u = g_u.values
        model_u = LinearRegression()
        model_u.fit(X_u, y_u)
        predicted_u = model_u.predict(X_u)
        residuals_u = y_u - predicted_u
        # pdb.set_trace()

        X_d = np.hstack((r_down_mean.to_numpy().reshape(-1, 1), r_1.to_numpy().reshape(-1, 1), r_2.to_numpy().reshape(-1$
        y_d = g_d.values
        model_d = LinearRegression()
        model_d.fit(X_d, y_d)
        predicted_d = model_d.predict(X_d)
        residuals_d = y_d - predicted_d
        # step 3: regression 截面回归计算因子值
        model = LinearRegression()
        model.fit(residuals_u, residuals_d)
        predicted_factor = model.predict(residuals_u)
        residuals_factor = residuals_d - predicted_factor

        # z = zscore(bp, winsorize_method='mad')
        # z = self.neutralize(z, style=['indzx','barra'])
        # zscore and store factor
        # print("Done for 1 day!")
        store_daily_tradable_vector(self.field_for_store, residuals_factor)



# tcd 时间重心偏离因子 月平滑: IC = 4.84
class tcd_monthly(TradeRule):
    def __init__(self,value):
        TradeRule.__init__(self,value)

    def trade(self):
        self.print_soy()
        factor = self.tcd.history_matrix(20)
        factor = np.mean(factor, axis = 1, keepdims = True)
        factor[~np.isfinite(factor)] = np.nan
        # pdb.set_trace()
        store_daily_tradable_vector(self.field_for_store, factor)


#### **Factor Backtesting System 因子回测系统(本地测试)**
##### incomplete!

In [5]:
# Import all libraries
import numpy as np
import pdb
import pandas as pd
from random import *
from scipy.stats import zscore as zs2
from sklearn.linear_model import LinearRegression, Perceptron
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import PercentFormatter
import time
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Import testing databases
# price_b1 = pd.read_csv('database/dictionary_close_b1.csv')
# volume_b1 = pd.read_csv('database/dictionary_volume_b1.csv')
ret_cc = pd.read_csv('database/dictionary_ret_cc.csv')


# Import factor databases
tcd = pd.read_csv("database/factor_tcd_monthly.csv")
amihud = pd.read_csv('database/factor_amihud.csv')
stability = pd.read_csv('database/factor_stability.csv')
real_skew = pd.read_csv('database/factor_real_skew_weekly.csv')
dictionary_lst = [tcd, amihud, stability, real_skew]


# Set all default variables
year_lst = ['2016', '2017', '2018', '2019', '2020', '2021', '2022']
factor_lst = [tcd, amihud, stability, real_skew]
factor_lst = df_preprocessing(factor_lst)

def df_preprocessing(factor_lst):
    for i in range(factor_lst):
        pass

def main(year_lst, factor_lst):
    print("Current Model: 单因子回测")
    return_lst = [1]
    return_lst2 = [1]
    drawdown_lst = []
    nav_lst = [[1], [1], [1], [1], [1]]
    nav_lst2 = [[0], [0], [0], [0], [0]]
    rank_ic_lst = []
    groupwise_returns = []
    groupwise_returns2 = []
    for year in range(len(year_lst)):
        this_year = year_lst[year]
        ICIR_lst = []
        val = backtesting(factor_lst, str(int(this_year) + 1))
        print("{}年化IC(日化均值):".format(int(this_year) + 1), round(val[0], 3))
        print("{}年化IR:".format(int(this_year) + 1), round(val[1], 3))
        for i in range(len(val[2][0][0])):
            rank_ic_lst.append(val[5][i]) 
            return_lst.append(return_lst[-1] * val[4][0][i])
            return_lst2.append(return_lst2[-1] + val[4][2][i])
            drawdown_lst.append(val[4][1][i])
            for j in range(5):
                nav_lst[j].append(nav_lst[j][-1] * val[2][0][j][i])
                nav_lst2[j].append(nav_lst2[j][-1] + val[2][1][j][i])
        groupwise_returns.append(val[3][0])
        groupwise_returns2.append(val[3][1])
        
    print("============================================")
    year_lst2 = year_lst[1:].copy()
    year_lst2.append(str(int(year_lst[-1]) + 1))
    plt_rank_ic(rank_ic_lst, model_name, year_lst, year_lst2)
    print('RankIC Plotting Finished!')
    print("============================================")
    
    plt_groupwise_nav(nav_lst, year_lst, year_lst2, "mul", model_name)
    plt_groupwise_returns(np.array(groupwise_returns), year_lst, year_lst2, 'mul', model_name)
    plt_top_bottom_portfolio(return_lst, drawdown_lst, year_lst, year_lst2, 'mul', model_name)
    print("Plotting Under Multiplication Algo Finished!")
    print("============================================")
    
    plt_groupwise_nav(nav_lst2, year_lst, year_lst2, 'add', model_name)
    plt_groupwise_returns(np.array(groupwise_returns2), year_lst, year_lst2, 'add', model_name)
    plt_top_bottom_portfolio(return_lst2, drawdown_lst, year_lst, year_lst2, 'add', model_name)
    print("Plotting Under Addition Algo Finished!")
    print("============================================")
    print('***End of Program***')


main(year_lst, factor_lst)


单因子回测:      Unnamed: 0  20160104  20160105  20160106  20160107  20160108  20160111  \
0             1  5.679305  5.915881  5.816259  6.157251  5.978985  6.136503   
1             2 -3.716650 -4.462886 -4.328103 -3.725256 -1.709296 -1.595593   
2             3  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
3             4 -3.534251 -4.108154 -4.039194 -3.108584 -2.699677 -2.954775   
4             5 -2.179091 -2.647789 -2.629647 -3.097545 -2.544033 -2.564873   
...         ...       ...       ...       ...       ...       ...       ...   
5493     csi500  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
5494     csi800  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
5495     cyb100  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
5496      kcb50  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
5497      sse50  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   

      20160112  20160113  20160114  ...   20

#### **Multi-factor Portfolio Backtesting System 多因子组合回测系统(server-side测试)**

In [ ]:
# Set Default Environment
from qpc import *
import numpy as np
import pdb
import pandas as pd
from random import *
from scipy.stats import zscore as zs2
from sklearn.linear_model import LinearRegression, Perceptron
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from keras.wrappers.scikit_learn import KerasRegressor
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.ticker import PercentFormatter
import time
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Initialization
Init('/qp/data/ini/cn.eq.base.ini')
set_now(20230721, 1500)

# set ramdom seed
random.seed(20230720)

# Import all binary datasets, transpose, rename; binary files should be generated by factor calculator (eg. factor/algo.py)
stability_df = readm2df('../../bins/formal/estu.a/stability_monthly.bin').T
amihud_df = readm2df('../../bins/formal/estu.a/amihud.bin').T
real_skew_df = readm2df('../../bins/formal/estu.a/real_skew_weekly.bin').T
corr_vrlag_df = readm2df('../../bins/formal/estu.a/corr_vrlag_monthly.bin').T
gjfz_df = readm2df('../../bins/formal/estu.a/gjfz_monthly.bin').T
gplj_df = readm2df('../../bins/formal/estu.a/gplj_monthly.bin').T
min_dev_df = readm2df('../../bins/formal/estu.a/min_dev4.bin').T
reth8_df = readm2df('../../bins/formal/estu.a/reth8_weekly.bin').T
ret_intraday_df = readm2df('../../bins/formal/estu.a/ret_intraday_monthly.bin').T

f1 = stability_df
f2 = amihud_df
f3 = real_skew_df
f4 = corr_vrlag_df
f5 = gjfz_df
f6 = gplj_df
f7 = min_dev_df
f8 = reth8_df
f9 = ret_intraday_df

# Define a list of years under consideration;
# year_lst can be modified, but it 1/ should be consecutive, 2/ should be in string type for later usage, and 3/ should be included by all the .bin creation time spans
year_lst = ['2016', '2017', '2018', '2019', '2020', '2021', '2022']

# Import all built-in datasets
volume = QPData('volume.b1')
price = QPData('close.b1')
open = QPData('open')
ret = QPData('ret.CC') # alternative : Day / 1000To10001d

def ICIR_calculator(current_factor, current_year, model_name): # Input: 单个因子的完整dataframe(包含所有日期)；string type的当前年份
    position = next((idx for idx, col in enumerate(current_factor.columns) if current_year in col[:4]), None)
    num = sum(1 for col in current_factor.columns if current_year in col)
    corr_lst = []
    for j in range(num):
        set_now(int(current_factor.columns[position + j]), 1500)
        factor_today = pd.DataFrame(zs2(current_factor.iloc[:, position + j].replace([np.inf, -np.inf, np.nan], 0)))
        ret_today = ret.history_matrix_pd(1)
        factor_today.index = current_factor.index
        common_index = factor_today.index.intersection(ret_today.index)
        factor_today = factor_today.loc[common_index, :]
        corr_lst.append(factor_today.iloc[:, 0].corr(ret_today.iloc[:, 0], method = 'spearman'))
    corr_lst = np.array(corr_lst)
    ICIR_val = abs(np.mean(corr_lst)) / np.std(corr_lst)
    return ICIR_val

def IC_calculator(weight, factor_lst, current_year, model_name):
    # 将所有的因子dataframe转成一样的shape
    current_year = str(int(current_year) + 1)
    common_index = set(factor_lst[0].index)
    for element in factor_lst[1:]:
        common_index = common_index & set(element.index)
    for ii in range(len(factor_lst)):
        factor_lst[ii] =  factor_lst[ii].loc[common_index]

    # 找到因子表中年份对应开始的日期的位置以及当年的交易日数量 
    position = next((idx for idx, col in enumerate(factor_lst[0].columns) if current_year in col[:4]), None)
    num = sum(1 for col in factor_lst[0].columns if current_year in col)
    corr_lst = []
    return_rate_lst = []
    return_rate_lst_add = []
    max_drawdown = []
    groupwise_nav_current = [[], [], [], [], []]
    groupwise_nav_add = [[], [], [], [], []]
    groupwise_returns_current = []
    groupwise_returns_add = []
    # 循环每个交易日
    for jj in tqdm(range(num), desc = "{}年IC计算进度".format(current_year)):
        set_now(int(factor_lst[0].columns[position + jj]), 1500)
        product_columns = []
        for i, df in enumerate(factor_lst):
            product_columns.append(pd.DataFrame(zs2(df.iloc[:, position + jj].replace([np.inf, -np.inf, np.nan], 0))) * weight[i])

        factor_today = pd.concat(product_columns, axis = 1).sum(axis = 1).to_frame()
        ret_today = ret.history_matrix_pd(1)
        factor_today.index = factor_lst[0].index
        common_index = factor_today.index.intersection(ret_today.index)
        factor_today = factor_today.loc[common_index, :]
        # corr_lst.append(np.corrcoef(factor_today.iloc[:, 0], ret_today.iloc[:, 0])[0, 1])　＃ alternative way to calculate correlation
        corr_lst.append(factor_today.iloc[:, 0].corr(ret_today.iloc[:, 0], method = 'spearman'))
        
        groupwise_nav_current = calc_groupwise_nav(factor_today, ret_today, groupwise_nav_current, choice = 'mul', model_name = model_name)  #(5, days)
        groupwise_nav_add = calc_groupwise_nav(factor_today, ret_today, groupwise_nav_add, choice = 'add', model_name = model_name)
        result = calc_groupwise_portfolio(return_rate_lst, max_drawdown, factor_today, ret_today, jj, return_rate_lst_add)
        return_rate_lst = result[0]
        max_drawdown = result[1]
        return_rate_lst_add = result[2]
    # 计算分组Returns
    groupwise_returns_current = calc_groupwise_returns(groupwise_nav_current, choice = 'mul') # 包含五个数的list类
    groupwise_returns_add = calc_groupwise_returns(groupwise_nav_add, choice = 'add')
    # 计算IC和ICIR
    corr_lst = np.array(corr_lst)
    IC = corr_lst.mean()
    IR = (corr_lst.mean() / corr_lst.std())
    return [IC, IR, [groupwise_nav_current, groupwise_nav_add], [groupwise_returns_current, groupwise_returns_add], [return_rate_lst, max_drawdown, return_rate_lst_add], corr_lst]

def ml_calculator(factor_lst, current_year, model_name):
    # 将所有的因子dataframe转成一样的shape
    common_index = set(factor_lst[0].index)
    for element in factor_lst[1:]:
        common_index = common_index & set(element.index)
    for ii in range(len(factor_lst)):
        factor_lst[ii] =  factor_lst[ii].loc[common_index]

    # 定位年份，求出一年的交易日数量
    position = next((idx for idx, col in enumerate(factor_lst[0].columns) if current_year in col[:4]), None)
    num = sum(1 for col in factor_lst[0].columns if current_year in col[:4])
    corr_lst = []
    
    return_rate_lst = []
    return_rate_lst_add = []
    max_drawdown = []
    groupwise_nav_current = [[], [], [], [], []]
    groupwise_nav_add = [[], [], [], [], []]
    groupwise_returns_current = []
    groupwise_returns_add = []
    
    mse_lst = []
    
    # 循环每个交易日
    for jj in tqdm(range(num-1), desc = "{}年IC计算进度".format(current_year)):
        # 算当天权重：
        set_now(int(factor_lst[0].columns[position + jj]), 1500)
        ret_today = ret.history_matrix_pd(1)
        x = pd.DataFrame(factor_lst[0].iloc[:, 0])
        for indexs in range(len(factor_lst) - 1):
            x = x.join(factor_lst[indexs].iloc[:, position + jj], rsuffix = 'df' + str(indexs))
        x.index = factor_lst[0].index
        common_index = x.index.intersection(ret_today.index)
        x_train = x.loc[common_index, :]
        y_train = ret_today.loc[common_index, :]
        x_train = x_train.replace([np.nan, np.inf, -np.inf], 0)
        y_train = y_train.replace([np.nan, np.inf, -np.inf], 0)
        if model_name == "ML LR":
            model = LinearRegression()
        elif model_name == "ML SVR":
            model = SVR(kernel = 'rbf', C = 1.0, epsilon = 0.05)
        elif model_name == "ML MLP":
            model = MLPRegressor(hidden_layer_sizes = (64, 32, 32, 16), max_iter = 500, activation = 'relu', solver = 'adam', alpha = 0.001)
        model.fit(x_train, y_train)
        # 用权重与第二天收益信息做corr，将corr储存到corr_lst
        set_now(int(factor_lst[0].columns[position + jj + 1]), 1500)
        x2 = pd.DataFrame(factor_lst[0].iloc[:, 0])
        ret_next = ret.history_matrix_pd(1) # 当日股票数 * 1, dataframe type
        for index2 in range(len(factor_lst)- 1):
            x2 = x2.join(factor_lst[index2].iloc[:, position + jj + 1], rsuffix = 'df' + str(index2))
        x2.index = factor_lst[0].index
        common_index = x2.index.intersection(ret_next.index)
        x_test = x2.loc[common_index, :]
        y_test = ret_next.loc[common_index, :]
        x_test = x_test.replace([np.nan, np.inf, -np.inf], 0)
        y_test = y_test.replace([np.nan, np.inf, -np.inf], 0)
        y_pred = model.predict(x_test)
        mse_lst.append(mean_squared_error(y_test, y_pred))
        corr_lst.append(pd.DataFrame(y_pred).iloc[:, 0].corr(y_test.iloc[:, 0].reset_index(drop = True), method = 'spearman'))
        factor_next = pd.DataFrame(y_pred)
        factor_next.index = common_index

        groupwise_nav_current = calc_groupwise_nav(factor_next, ret_next, groupwise_nav_current, choice = 'mul', model_name = model_name)
        groupwise_nav_add = calc_groupwise_nav(factor_next, ret_next, groupwise_nav_add, choice = 'add', model_name = model_name)
        result = calc_groupwise_portfolio(return_rate_lst, max_drawdown, factor_next, ret_next, jj, return_rate_lst_add)
        return_rate_lst = result[0]
        max_drawdown = result[1]
        return_rate_lst_add = result[2]
    
    groupwise_returns_current = calc_groupwise_returns(groupwise_nav_current, choice = 'mul') # 包含五个数的list类
    groupwise_returns_add = calc_groupwise_returns(groupwise_nav_add, choice = 'add')
    
    print("{}-{}-MSE均值: ".format(current_year, model_name), np.array(mse_lst).mean())
    
    # 计算IC和ICIR
    corr_lst = np.array(corr_lst)
    IC = corr_lst.mean()
    IR = (corr_lst.mean() / corr_lst.std())
    return [IC, IR, [groupwise_nav_current, groupwise_nav_add], [groupwise_returns_current, groupwise_returns_add], [return_rate_lst, max_drawdown, return_rate_lst_add], corr_lst]

def lstm_factor_predict(factor_lst, year_lst, model_name):

    common_index = set(factor_lst[0].index)
    for element in factor_lst[1:]:
        common_index = common_index & set(element.index)
    for ii in range(len(factor_lst)):
        factor_lst[ii] =  factor_lst[ii].loc[common_index]

    num_stocks_total = (factor_lst[0].shape)[0]
    num_days_total = (factor_lst[0].shape)[1]
    test_size = 618

    pred_dict = {}

    def create_time_series_dataset(data, day_past):
        data_x = []
        data_y = []
        for j in range(day_past, len(data)):
            data_x.append(data[j - day_past:j, 0:data.shape[1]])
            data_y.append(data[j, 0])
        return np.array(data_x), np.array(data_y)

    for i in range(num_stocks_total): # 对每一只股票做LSTM
        stock_index = factor_lst[0].index[i]
        df_for_train = pd.concat([factor_lst[0].iloc[i, :-test_size], factor_lst[1].iloc[i, :-test_size], factor_lst[2].iloc[i, :-test_size], factor_lst[3].iloc[i, :-test_size], factor_lst[4].iloc[i, :-test_size], factor_lst[5].iloc[i, :-test_size], factor_lst[6].iloc[i, :-test_size], factor_lst[7].iloc[i, :-test_size], factor_lst[8].iloc[i, :-test_size]], axis = 0)
        df_for_train = df_for_train.T
        df_for_test = pd.concat([factor_lst[0].iloc[i, -test_size:], factor_lst[1].iloc[i, -test_size:], factor_lst[2].iloc[i, -test_size:], factor_lst[3].iloc[i, -test_size:], factor_lst[4].iloc[i, -test_size:], factor_lst[5].iloc[i, -test_size:], factor_lst[6].iloc[i, -test_size:], factor_lst[7].iloc[i, -test_size:], factor_lst[8].iloc[i, -test_size:]], axis = 0)
        df_for_test = df_for_test.T
        scaler = MinMaxScaler(feature_range = (0, 1))
        df_for_train_scaled = scaler.fit_transform(df_for_train) # shape = ((days-test_size) * 9)
        df_for_test_scaled = scaler.fit_transform(df_for_test)
        trainX, trainY = create_time_series_dataset(df_for_train_scaled, 20) # 月频学习
        testX, testY = create_time_series_dataset(df_for_test_scaled, 20)
        pdb.set_trace()
        def build_model(optimizer):
            grid_model = Sequential()
            grid_model.add(LSTM(50, return_sequences = True, input_shape = (20, 9)))
            grid_model.add(LSTM(50))
            grid_model.add(Dropout(0.2))
            grid_model.add(Dense(1))

            grid_model.compile(loss = 'mse',optimizer = optimizer)
            return grid_model

        model = KerasRegressor(build_fn = build_model, verbose = 1, validation_data = (testX, testY)) # verbose = 0 以省略冗长输出
        parameters = {'batch_size' : [16,20],
                    'epochs' : [8,10],
                    'optimizer' : ['adam','Adadelta'] }

        grid_search  = GridSearchCV(estimator = model,
                                    param_grid = parameters,
                                    cv = 2)
        grid_search = grid_search.fit(trainX,trainY)
        my_model = grid_search.best_estimator_.model
        prediction = my_model.predict(testX) # shape (500, 1)
        pred_dict[stock_index] = prediction.flatten()
        factor_predicted_df = pd.DataFrame(pred_dict).T # shape (stock_num, test_size)

        corr_lst = []
        return_rate_lst = []
        return_rate_lst_add = []
        max_drawdown = []
        groupwise_nav_current = [[], [], [], [], []]
        groupwise_nav_add = [[], [], [], [], []]
        groupwise_returns_current = []
        groupwise_returns_add = []
        for k in range(test_size): # 对于每一天求corr等
            set_now(int(factor_lst[0].columns[-test_size + k]), 1500)

            factor_today = factor_predicted_df.iloc[:, k]
            ret_today = ret.history_matrix_pd(1)
            common_index = factor_today.index.intersection(ret_today.index)
            factor_today = factor_today.loc[common_index, :]
            corr_lst.append(factor_today.iloc[:, 0].corr(ret_today.iloc[:, 0], method = 'spearman'))
            
            groupwise_nav_current = calc_groupwise_nav(factor_today, ret_today, groupwise_nav_current, choice = 'mul', model_name = model_name)  #(5, days)
            groupwise_nav_add = calc_groupwise_nav(factor_today, ret_today, groupwise_nav_add, choice = 'add', model_name = model_name)
            result = calc_groupwise_portfolio(return_rate_lst, max_drawdown, factor_today, ret_today, return_rate_lst_add)
            return_rate_lst = result[0]
            max_drawdown = result[1]
            return_rate_lst_add = result[2]
        # 计算分组Returns
        groupwise_returns_current = calc_groupwise_returns(groupwise_nav_current, choice = 'mul') # 包含五个数的list类
        groupwise_returns_add = calc_groupwise_returns(groupwise_nav_add, choice = 'add')
        # 计算IC和ICIR
        corr_lst = np.array(corr_lst)
        IC = corr_lst.mean()
        IR = (corr_lst.mean() / corr_lst.std())
        return [IC, IR, [groupwise_nav_current, groupwise_nav_add], [groupwise_returns_current, groupwise_returns_add], [return_rate_lst, max_drawdown, return_rate_lst_add], corr_lst]c

#############################################################################################################################################################################################

def drawdown_calculator(asset_values):
    max_drawdown = 0
    peak = asset_values[0]
    for value in asset_values:
        if value > peak:
            peak = value
        else: 
            drawdown = (peak - value) / peak
            if drawdown > max_drawdown:
                max_drawdown = drawdown
    return max_drawdown

def calc_groupwise_nav(factor_today, ret_today, groupwise_nav_current, choice, model_name):
    epsilon = factor_today.iloc[:, 0].mean(axis = 0) / 10000
    # 计算return：找到factor_today里的五组，将其编号对应到ret_today，分别算出每一组的return
    factor_today['rank_percentage'] = factor_today.iloc[:, 0].rank(pct = True)
    if model_name == "ML MLP":
        factor_today.iloc[:, 0] += np.random.rand(factor_today.shape[0]) * epsilon
    group1 = factor_today[factor_today['rank_percentage'] >= 0.8]
    group1_indices = group1.index 
    return_1 = ret_today.loc[group1_indices, :].sum()
    
    group2 = factor_today[(factor_today['rank_percentage'] < 0.8) & (factor_today['rank_percentage'] >= 0.6)]
    group2_indices = group2.index 
    return_2 = ret_today.loc[group2_indices, :].sum()
    
    group3 = factor_today[(factor_today['rank_percentage'] < 0.6) & (factor_today['rank_percentage'] >= 0.4)]
    group3_indices = group3.index 
    return_3 = ret_today.loc[group3_indices, :].sum()
    
    group4 = factor_today[(factor_today['rank_percentage'] < 0.4) & (factor_today['rank_percentage'] >= 0.2)]
    group4_indices = group4.index 
    return_4 = ret_today.loc[group4_indices, :].sum()
    
    group5 = factor_today[factor_today['rank_percentage'] < 0.2]
    group5_indices = group5.index 
    return_5 = ret_today.loc[group5_indices, :].sum()
    
    if choice == 'mul':
        groupwise_nav_current[0].append((1 + float(return_1) / (10000 * len(group1_indices))))
        groupwise_nav_current[1].append((1 + float(return_2) / (10000 * len(group2_indices))))
        groupwise_nav_current[2].append((1 + float(return_3) / (10000 * len(group3_indices))))
        groupwise_nav_current[3].append((1 + float(return_4) / (10000 * len(group4_indices))))
        groupwise_nav_current[4].append((1 + float(return_5) / (10000 * len(group5_indices))))
    elif choice == 'add':
        groupwise_nav_current[0].append(float(return_1) / (10000 * len(group1_indices)))
        groupwise_nav_current[1].append(float(return_2) / (10000 * len(group2_indices)))
        groupwise_nav_current[2].append(float(return_3) / (10000 * len(group3_indices)))
        groupwise_nav_current[3].append(float(return_4) / (10000 * len(group4_indices)))
        groupwise_nav_current[4].append(float(return_5) / (10000 * len(group5_indices)))
    
    return groupwise_nav_current

def calc_groupwise_returns(groupwise_nav_current, choice):
    if choice == 'mul':
        new_return = []
        for k in range(5):
            multiple = 1
            for kk in range(len(groupwise_nav_current[k])):
                multiple = multiple * groupwise_nav_current[k][kk]
            new_return.append(multiple - 1)
    elif choice == 'add':
        new_return = []
        for k in range(5):
            addition = np.array(groupwise_nav_current[k]).sum()
            new_return.append(addition)
    return new_return

def calc_groupwise_portfolio(return_rate_lst, max_drawdown, factor_today, ret_today, return_rate_lst_add):
    # 计算return：找到factor_today里的第一组和第五组，将其编号对应到ret_today，分别算出第一组和第五组的return值然后做减
    factor_today['rank_percentage'] = factor_today.iloc[:, 0].rank(pct = True)
    top_20_percent = factor_today[factor_today['rank_percentage'] >= 0.8]
    top_20_indices = top_20_percent.index 
    bottom_20_percent = factor_today[factor_today['rank_percentage'] <= 0.2]
    bottom_20_indices = bottom_20_percent.index
    
    return_top_20 = ret_today.loc[top_20_indices, :].sum()
    return_bottom_20 = ret_today.loc[bottom_20_indices, :].sum()
    return_rate_lst.append(1 + float(return_top_20 - return_bottom_20) / (10000 * (len(top_20_indices) + len(bottom_20_indices))))
    return_rate_lst_add.append(float(return_top_20 - return_bottom_20) / (10000 * (len(top_20_indices) + len(bottom_20_indices))))
    
    # 计算回撤: (多头情况下)找到第一组和第五组的股票代码，去close表里算出每一分钟当前选股/资产配置下的总资产，append到asset_lst
    total_index = factor_today[(factor_today['rank_percentage'] <= 0.2) | (factor_today['rank_percentage'] >= 0.8)].index
    asset_lst = price.daily_matrix_pd().loc[total_index, :].sum(axis = 0)
    max_drawdown.append(drawdown_calculator(asset_lst))
    return [return_rate_lst, max_drawdown, return_rate_lst_add]

def plt_rank_ic(rank_ic_lst, model_name, year_lst, year_lst2):
    days = list(range(1, len(rank_ic_lst) + 1))
    plt.figure(figsize = (12, 8))
    
    plt.bar(days, rank_ic_lst, alpha = 0.5, label = 'DailyRankIC', color = 'blue')
    plt.axhline(y = np.array(rank_ic_lst).mean(), color = 'orange', linestyle = 'dashed', label = 'mean = {}'.format(round(np.array(rank_ic_lst).mean(), 4)))
    plt.ylabel('Weighted RankIC')
    plt.xlabel('Date')
    count = False
    for line_x in days[::240]:
        plt.axvline(x = line_x, alpha = 0.5, linestyle = 'dotted')
        if (len(days) - line_x) >= 230:
            pos = int((line_x * 2 + 240) / 2)
            y = np.array(rank_ic_lst[pos : pos + 240]).mean()
            if count == False:
                plt.plot(pos, y, marker = 'o', color = 'red', markersize = '6', markerfacecolor = None, label = 'Annual Mean')
                count = True
            else:
                plt.plot(pos, y, marker = 'o', color = 'red', markersize = '6', markerfacecolor = None)
        else: 
            pos = int((len(days) + line_x) / 2)
            y = np.array(rank_ic_lst[pos:]).mean()
            if count == False:
                plt.plot(pos, y, marker = 'o', color = 'red', markersize = '6', markerfacecolor = None, label = 'Annual Mean')
            else:
                plt.plot(pos, y, marker = 'o', color = 'red', markersize = '6', markerfacecolor = None)
    if  model_name == 'ICIR Weight':
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst], rotation = 15)
    elif (model_name == 'ML LR') or (model_name =="ML SVR") or (model_name == "ML MLP"):
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst2], rotation = 15)
    elif model_name == "ML LSTM":
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst2[-3:]], rotation = 15)
    plt.title("[Multi-factor] Weighted RankIC (H = 1)")
    plt.grid(True, linestyle = 'dashed', alpha = 0.5)
    plt.legend()
    plt.savefig("figures/[{}]rank_ic.png".format(model_name))
    print("RankIC Plotting Finished!")
    
def plt_groupwise_nav(nav_lst, year_lst, year_lst2, choice, model_name):
    days = list(range(1, len(nav_lst[0])))
    plt.figure(figsize = (12, 8))
    nav_lst *= 100
    plt.plot(days, nav_lst[0][1:], label = 'Group 1', color = 'blue')
    plt.plot(days, nav_lst[1][1:], label = 'Group 2', color = 'red')
    plt.plot(days, nav_lst[2][1:], label = 'Group 3', color = 'green')
    plt.plot(days, nav_lst[3][1:], label = 'Group 4', color = 'orange')
    plt.plot(days, nav_lst[4][1:], label = 'Group 5', color = 'purple')
    plt.xlabel('Years')
    plt.ylabel('NAV')
    plt.title('[Multi-factor] Groupwise NAV (H = 1)')
    plt.legend()
    # plt.twinx()
    # plt.legend(bbox_to_anchor = (1, 1), loc = 'upper right')
    for line_x in days[::240]:
        plt.axvline(x = line_x, alpha = 0.5, linestyle = 'dotted')
    if  model_name == 'ICIR Weight':
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst], rotation = 15)
    elif (model_name == 'ML LR') or (model_name =="ML SVR") or (model_name == "ML MLP"):
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst2], rotation = 15)
    elif model_name == "ML LSTM":
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst2[-3:]], rotation = 15)
    plt.grid(True, linestyle = 'dashed', alpha = 0.5)
    plt.savefig("figures/[{}]groupwise_nav_{}.png".format(model_name, choice))
    print("Groupwise NAV Plotting Finished!")

def plt_groupwise_returns(data, year_lst, year_lst2, choice, model_name):
    if  model_name == 'ICIR Weight':
        years = np.arange(int(year_lst[0]) + 1, int(year_lst[-1]) + 2)
    elif (model_name == 'ML LR') or (model_name =="ML SVR") or (model_name == "ML MLP"):
        years = np.arange(int(year_lst2[0]) + 1, int(year_lst2[-1]) + 2)
    elif model_name == "ML LSTM":
        years = np.arange(int(year_lst2[-3]) + 1, int(year_lst2[-1]) + 2)
    groups = ['Group 1', 'Group 2', 'Group 3', 'Group 4', 'Group 5']
    colors = ['purple', 'orange', 'green', 'red', 'blue']
    bar_width = 0.15
    x_position = np.arange(len(years))
    plt.figure(figsize = (15, 6))
    for i in range(len(groups)):
        plt.bar(x_position + i * bar_width, data[:, 4-i], width = bar_width, label = groups[i], color = colors[i])
        
    plt.xlabel('Years')
    plt.ylabel('Return')
    plt.title('[Multi-factor] Groupwise Returns')
    plt.xticks(x_position + (len(groups) - 1) * bar_width / 2, years, rotation = 15)
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax = 1, decimals = 0))
    if model_name == 'ICIR Weight':
        year_lst_int = [str(int(item) + 1) if item.isdigit() else item for item in year_lst]
    elif (model_name == 'ML LR') or (model_name =="ML SVR") or (model_name == "ML MLP"):
        year_lst_int = [str(int(item) + 1) if item.isdigit() else item for item in year_lst2]
    elif model_name == "ML LSTM":
        year_lst_int = [str(int(item) + 1) if item.isdigit() else item for item in year_lst2[-3:]]
    for line_x in year_lst_int:
        plt.axvline(x = line_x, alpha = 0.5, linestyle = 'dotted')
    plt.grid(True, linestyle = 'dashed', alpha = 0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig("figures/[{}]groupwise_returns_{}.png".format(model_name, choice))
    print("Groupwise Returns Plotting Finished!")

def plt_top_bottom_portfolio(return_lst, drawdown_lst, year_lst, year_lst2, choice, model_name):
    days = list(range(1, len(return_lst)))
    plt.figure(figsize = (12, 8))
    plt.plot(days, return_lst[1:], label = 'Cummulative Returns', color = 'blue')
    plt.xlabel('Years')
    plt.ylabel('Cummulative Return')
    plt.title('[Multi-factor] Top-bottom Portfolio')
    plt.legend()
    
    plt.twinx()
    bottom_values = [-value for value in drawdown_lst]
    plt.bar(days, drawdown_lst, alpha = 0.5, label = 'Max Drawdown', color = 'red', bottom = bottom_values)
    plt.ylabel('Max Drawdown')
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax = 1, decimals = 0))
    
    plt.legend(bbox_to_anchor = (1, 1), loc = 'upper right')
    for line_x in days[::240]:
        plt.axvline(x = line_x, alpha = 0.5, linestyle = 'dotted')
    if  model_name == 'ICIR Weight':
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst], rotation = 15)
    elif (model_name == 'ML LR') or (model_name =="ML SVR") or (model_name == "ML MLP"):
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst2], rotation = 15)
    elif model_name == "ML LSTM":
        plt.xticks(days[::240], [str(int(last_year) + 1) for last_year in year_lst2[-3:]], rotation = 15)
    plt.grid(True, linestyle = 'dashed', alpha = 0.5)
    plt.savefig("figures/[{}]portfolio_{}.png".format(model_name, choice))
    print("Portfolio Plotting Finished!")

# ICIR加权算法: 2016-2023年化IC均值达7.2
# ML算法: LinearRegression，Perceptron, SVR
def main(model_name, year_lst):
    print("Current Model: {}".format(model_name))
    return_lst = [1]
    return_lst2 = [1]
    drawdown_lst = []
    nav_lst = [[1], [1], [1], [1], [1]]
    nav_lst2 = [[0], [0], [0], [0], [0]]
    rank_ic_lst = []
    groupwise_returns = []
    groupwise_returns2 = []
    if model_name != "ML LSTM":
        for year in range(len(year_lst)):
            this_year = year_lst[year]
            ICIR_lst = []
            factor_lst = [f1, f2, f3, f4, f5, f6, f7, f8, f9]
            
            if model_name == "ICIR Weight":
                for i in range(len(factor_lst)):
                    ICIR_lst.append(ICIR_calculator(factor_lst[i], this_year, model_name)) # ICIR_lst最后会为一个 len = 9 并储存每单个因子年化/季化ICIR的array
                weight = ICIR_lst
                print("{}年化多因子权重(ICIR比值): {}".format(this_year, np.round(np.array(weight), 3)))
                val = IC_calculator(weight, factor_lst, this_year, model_name)
            elif (model_name == 'ML LR') or (model_name =="ML SVR") or (model_name == "ML MLP"):
                val = ml_calculator(factor_lst, str(int(this_year) + 1), model_name)
            elif model_name == "ML LSTM":
                pass  ## pending to be written
            print("{}年化IC(日化均值):".format(int(this_year) + 1), round(val[0], 3))
            print("{}年化IR:".format(int(this_year) + 1), round(val[1], 3))
            for i in range(len(val[2][0][0])):
                rank_ic_lst.append(val[5][i]) 
                return_lst.append(return_lst[-1] * val[4][0][i])
                return_lst2.append(return_lst2[-1] + val[4][2][i])
                drawdown_lst.append(val[4][1][i])
                for j in range(5):
                    nav_lst[j].append(nav_lst[j][-1] * val[2][0][j][i])
                    nav_lst2[j].append(nav_lst2[j][-1] + val[2][1][j][i])
            groupwise_returns.append(val[3][0])
            groupwise_returns2.append(val[3][1])
            
        if model_name == "ICIR Weight":
            pass
        elif (model_name == "ML LR") or (model_name == "ML SVR") or (model_name == "ML MLP"):
            year_lst.insert(0, str(int(year_lst[0]) - 1))
            year_lst = year_lst[:-1]
            
        print("============================================")
        year_lst2 = year_lst[1:].copy()
        year_lst2.append(str(int(year_lst[-1]) + 1))
        plt_rank_ic(rank_ic_lst, model_name, year_lst, year_lst2)
        print('RankIC Plotting Finished!')
        print("============================================")
        
        plt_groupwise_nav(nav_lst, year_lst, year_lst2, "mul", model_name)
        plt_groupwise_returns(np.array(groupwise_returns), year_lst, year_lst2, 'mul', model_name)
        plt_top_bottom_portfolio(return_lst, drawdown_lst, year_lst, year_lst2, 'mul', model_name)
        print("Plotting Under Multiplication Algo Finished!")
        print("============================================")
        
        plt_groupwise_nav(nav_lst2, year_lst, year_lst2, 'add', model_name)
        plt_groupwise_returns(np.array(groupwise_returns2), year_lst, year_lst2, 'add', model_name)
        plt_top_bottom_portfolio(return_lst2, drawdown_lst, year_lst, year_lst2, 'add', model_name)
        print("Plotting Under Addition Algo Finished!")
        print("============================================")
        print('***End of Program***')
    else:
        # LSTM Program
        val = lstm_factor_predict(factor_lst, year_lst, model_name)
        for i in range(len(val[2][0][0])):
            rank_ic_lst.append(val[5][i]) 
            return_lst.append(return_lst[-1] * val[4][0][i])
            return_lst2.append(return_lst2[-1] + val[4][2][i])
            drawdown_lst.append(val[4][1][i])
            for j in range(5):
                nav_lst[j].append(nav_lst[j][-1] * val[2][0][j][i])
                nav_lst2[j].append(nav_lst2[j][-1] + val[2][1][j][i])
            groupwise_returns.append(val[3][0])
            groupwise_returns2.append(val[3][1])
        print("500天IC(日化均值):", round(val[0], 3))
        print("500天IR:", round(val[1], 3))
        
        if model_name == "ICIR Weight":
            pass
        elif (model_name == "ML LR") or (model_name == "ML SVR") or (model_name == "ML MLP") or (model_name == "ML LSTM"):
            year_lst.insert(0, str(int(year_lst[0]) - 1))
            year_lst = year_lst[:-1]

        print("============================================")
        year_lst2 = year_lst[1:].copy()
        year_lst2.append(str(int(year_lst[-1]) + 1))
        plt_rank_ic(rank_ic_lst, model_name, year_lst, year_lst2)
        print('RankIC Plotting Finished!')
        print("============================================")
        
        plt_groupwise_nav(nav_lst, year_lst, year_lst2, "mul", model_name)
        plt_groupwise_returns(np.array(groupwise_returns), year_lst, year_lst2, 'mul', model_name)
        plt_top_bottom_portfolio(return_lst, drawdown_lst, year_lst, year_lst2, 'mul', model_name)
        print("Plotting Under Multiplication Algo Finished!")
        print("============================================")
        
        plt_groupwise_nav(nav_lst2, year_lst, year_lst2, 'add', model_name)
        plt_groupwise_returns(np.array(groupwise_returns2), year_lst, year_lst2, 'add', model_name)
        plt_top_bottom_portfolio(return_lst2, drawdown_lst, year_lst, year_lst2, 'add', model_name)
        print("Plotting Under Addition Algo Finished!")
        print("============================================")
        print('***End of Program***')

# 这里提供了五种模型可以选择
# ICIR Weight为年化ICIR加权；
# ML LR为Linear Regression多变量线性回归，ML SVR为Support Vector Regression支持向量回归；这两种是常见的监督学习
# ML MLP为Multi-Layer Perceptron多层感知机，ML RNN为Recurrent Neural Network循环神经网络，多用于时序信息的机器学习；这两种是常见的深度学习
model_name = ["ICIR Weight", "ML LR", "ML SVR", "ML MLP", "ML RNN"] 
main(model_name[3], year_lst)

<div style = "text-align: center">
    <div style = "display: flex; justify-content: space-between;">
        <p style = "width: 24%; text-align: center;"> ICIR Weighted </p>
        <p style = "width: 24%; text-align: center;"> ML LR </p>
        <p style = "width: 24%; text-align: center;"> ML SVR </p>
        <p style = "width: 24%; text-align: center;"> ML MLP </p>
    </div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]running_result.jpg" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]running_result.jpg" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]running_result.jpg" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]running_result.jpg" width = "23%" style = "">
    </div>
    <div height = "25px"></div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]rank_ic.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]rank_ic.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]rank_ic.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]rank_ic.png" width = "23%" style = "">
    </div>
    <div height = "25px"></div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]groupwise_nav_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]groupwise_nav_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]groupwise_nav_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]groupwise_nav_add.png" width = "23%" style = "">
    </div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]groupwise_returns_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]groupwise_returns_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]groupwise_returns_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]groupwise_returns_add.png" width = "23%" style = "">
    </div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]portfolio_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]portfolio_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]portfolio_add.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]portfolio_add.png" width = "23%" style = "">
    </div>
    <div height = "25px"></div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]groupwise_nav_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]groupwise_nav_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]groupwise_nav_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]groupwise_nav_mul.png" width = "23%" style = "">
    </div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]groupwise_returns_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]groupwise_returns_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]groupwise_returns_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]groupwise_returns_mul.png" width = "23%" style = "">
    </div>
    <div style = "display: flex; flex-wrap: nowrap;">
        <img src = "figures/[ICIR Weight]portfolio_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML LR]portfolio_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML SVR]portfolio_mul.png" width = "23%" style = "margin-right: 2%;">
        <img src = "figures/[ML MLP]portfolio_mul.png" width = "23%" style = "">
    </div>
</div>

#### 对上述图标的解释

- 每个算法下共有8张图
- 第一张图`[Algo name]running_result`为运行代码产生的output，展示了中间步骤的一些输出指标 以及模型运行效率（MSE）
- 第二张图`[Algo name]rank_ic`包括因子每天的rank ic值，年化rank ic均值，以及2016年至2023年rank ic均值
- 第3-5张图为`“因子每日收益/收益率做累计加法”`算法下的groupwise表现图；我们只把每天的增幅跌幅做累加，换言之 每天我们默认投入每一组的资金相等（为总资本1/5）
- 第6-8张图为`“因子每日收益/收益率做累计乘法”`算法下的groupwise表现图；换言之，若第一组的本金在第一天减少了，则第二天的投入也会减少（越来越少但是不会亏到负）

#### 发现/结论/思考

- 根据ICIR加权的Groupwise_returns图发现，五组收益相较于头组负收益较少，原因可能是单因子对于价格下跌的预测好于对价格上涨的预测，或单因子有更大把成功握预测下跌情况（相较预测上涨情况）；
    因此猜测，可以用性能相反的因子来做合成因子以提高头组和五组的总收益（eg. A因子预测价格上涨的能力与准确性更强，B因子预测价格下跌的能力与准确性更强）；
    （猜想）或许可以通过因子值的skewness以及设定置信区间下的统计检验 来判断某因子对于涨幅和跌幅预测的信心和强度。
- 根据Groupwise_returns图标发现，在Machine Learning三种方法下的五组收益显著提高，在MLP算法下的收益累计(add)于2016年可超过60%（收益累乘(mul)于2016年可达80%）
- SVR算法下的扰动较大；ICIR加权算法下扰动较小。ML算法扰动较大的原因主要因为我们每天进行一次ML的拟合来计算第二天的因子，而ICIR加权的权重在一年内的计算里不发生变化
- 三种ML算法下的mean square error相近，均约在5w-9w区间内，其中SVR的表现相对最优（也是其扰动较大的原因之一）
- RNN/LSTM算法有两种方法：可以利用每只股票过去20天的交易指数（开盘价、收盘价、交易量等）对股票价格进行学习，来预测当天的价格情况，根据预测的涨跌来选股；也可以利用已有的多个单因子过去20天的因子值对过去20天的收益率指数进行学习，来预测当天的收益指数作为组合因子的值，从而进行选股
